# ISP Feature Impact Analysis — Correlation & Statistical Evidence

This notebook provides **data-driven evidence** that ISP forecast revision features carry actionable information for IDA price forecasting. Analysis covers all three test periods.

Key questions:
1. How correlated are ISP revisions with the IDA-DAM price spread?
2. Does this correlation change across forecasting horizons (t+1h, t+2h, t+3h)?
3. How does the relationship evolve from winter to spring?
4. How do ISP features compare to BM and price features in predictive power?

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

FILENAME = '/content/Final2026_with_ENTSOE.csv'
try:
    df = pd.read_csv(FILENAME, sep=',')
    if len(df.columns) == 1: df = pd.read_csv(FILENAME, sep=';')
except: df = pd.read_csv(FILENAME)

df = df.rename(columns={
    'Day time': 'DELIVERY_MTU', 'DAM price': 'DAM_MCP',
    'IDA1': 'MCP', 'IDA2': 'MCP_IDA2', 'IDA3': 'MCP_IDA3',
    'BM price': 'BM_IMBALANCE_PRICE', 'BMmDAM price': 'BMmDAMMCP',
})
df['DELIVERY_MTU'] = pd.to_datetime(df['DELIVERY_MTU'])
df = df.sort_values('DELIVERY_MTU').reset_index(drop=True)
df['hour'] = df['DELIVERY_MTU'].dt.hour

# Fill missing
df['MCP_IDA3'] = df['MCP_IDA3'].fillna(0.0)
df['MCP_IDA2'] = df['MCP_IDA2'].ffill().bfill().fillna(0.0)
df['MCP'] = df['MCP'].ffill().bfill().fillna(df['MCP_IDA2'])
df['DAM_MCP'] = df['DAM_MCP'].ffill().bfill()
df['BM_IMBALANCE_PRICE'] = df['BM_IMBALANCE_PRICE'].ffill().bfill().fillna(df['BM_IMBALANCE_PRICE'].median())
df['ISP2_RES'] = df['ISP2_RES'].ffill().bfill().fillna(df['ISP1_RES'])
df['ISP2_Load'] = df['ISP2_Load'].ffill().bfill().fillna(df['ISP1_Load'])

# ISP features
df['DA_RES'] = df['Wind_DA_forecast'] + df['Solar_DA_forecast']
is_ida3 = df['hour'] >= 12
df['ida3_available'] = (df['MCP_IDA3'] != 0).astype(float)
df['IDA_target'] = np.where(df['ida3_available'] == 1, df['MCP_IDA3'], df['MCP_IDA2'])
df['latest_RES'] = np.where(is_ida3, df['ISP3_RES'], df['ISP2_RES'])
df['latest_Load'] = np.where(is_ida3, df['ISP3_Load'], df['ISP2_Load'])
df['latest_RES_revision'] = df['latest_RES'] - df['DA_RES']
df['latest_Load_revision'] = df['latest_Load'] - df['SystemLoad_DA_forecast']
df['latest_net_load'] = df['latest_Load'] - df['latest_RES']
df['DA_net_load'] = df['SystemLoad_DA_forecast'] - df['DA_RES']
df['latest_net_load_revision'] = df['latest_net_load'] - df['DA_net_load']
df['RES_revision_ISP2_to_ISP3'] = np.where(is_ida3, df['ISP3_RES'] - df['ISP2_RES'], 0.0)
df['Load_revision_ISP2_to_ISP3'] = np.where(is_ida3, df['ISP3_Load'] - df['ISP2_Load'], 0.0)

# BM features
df['bm_roll_mean_4'] = df['BM_IMBALANCE_PRICE'].rolling(4, min_periods=1).mean()
df['bm_roll_std_4'] = df['BM_IMBALANCE_PRICE'].rolling(4, min_periods=1).std().fillna(0)
df['bm_roll_mean_12'] = df['BM_IMBALANCE_PRICE'].rolling(12, min_periods=1).mean()
df['bm_ida_spread'] = df['BM_IMBALANCE_PRICE'] - df['MCP_IDA2']
df['ida_momentum'] = df['MCP_IDA2'].diff(4).fillna(0)

# Targets at different horizons
for h_steps, h_name in [(4, '1h'), (8, '2h'), (12, '3h')]:
    df[f'IDA_target_{h_name}'] = df['IDA_target'].shift(-h_steps)

# Spread = IDA - DAM (what ISP revisions should predict)
df['IDA_DAM_spread'] = df['IDA_target'] - df['DAM_MCP']
for h_steps, h_name in [(4, '1h'), (8, '2h'), (12, '3h')]:
    df[f'spread_{h_name}'] = df[f'IDA_target_{h_name}'] - df['DAM_MCP']

df = df.dropna(subset=['IDA_target_1h', 'IDA_target_2h', 'IDA_target_3h']).reset_index(drop=True)
df = df[df['DELIVERY_MTU'] >= '2025-10-02'].copy().reset_index(drop=True)

print(f"Dataset: {df.shape}")
print(f"Period: {df['DELIVERY_MTU'].min()} to {df['DELIVERY_MTU'].max()}")

## 1. Correlation Matrix: All Features vs Targets

Heatmap showing Pearson correlation of each feature with the IDA target at t+1h, t+2h, t+3h, and the IDA-DAM spread.

In [ ]:
# Define feature groups for organized display
ISP_FEATURES = [
    'latest_RES', 'latest_Load', 'latest_RES_revision', 'latest_Load_revision',
    'latest_net_load_revision', 'latest_net_load',
    'RES_revision_ISP2_to_ISP3', 'Load_revision_ISP2_to_ISP3',
]
BM_FEATURES = [
    'BM_IMBALANCE_PRICE', 'bm_roll_mean_4', 'bm_roll_std_4', 'bm_roll_mean_12', 'bm_ida_spread',
]
PRICE_FEATURES = ['MCP', 'MCP_IDA2', 'DAM_MCP']
OTHER_FEATURES = [
    'SystemLoad_DA_forecast', 'Wind_DA_forecast', 'Solar_DA_forecast',
    'hour', 'ida_momentum',
]

ALL_FEATURES = PRICE_FEATURES + ISP_FEATURES + BM_FEATURES + OTHER_FEATURES
TARGET_COLS = ['IDA_target_1h', 'IDA_target_2h', 'IDA_target_3h',
               'spread_1h', 'spread_2h', 'spread_3h']

# Period definitions
PERIODS = {
    'A: Winter (Oct-Jan)': ('2025-10-02', '2026-01-18'),
    'B: Full (Oct-Mar)':   ('2025-10-02', '2026-03-12'),
    'C: Spring (Jan-Mar)': ('2026-01-20', '2026-03-12'),
}

In [ ]:
# Full correlation heatmap per period
fig, axes = plt.subplots(1, 3, figsize=(24, 12))

for ax, (pname, (d_start, d_end)) in zip(axes, PERIODS.items()):
    mask = (df['DELIVERY_MTU'] >= d_start) & (df['DELIVERY_MTU'] <= d_end + ' 23:45:00')
    sub = df[mask].copy()

    corr = sub[ALL_FEATURES + TARGET_COLS].corr().loc[ALL_FEATURES, TARGET_COLS]

    # Color-code row labels
    colors = []
    for f in ALL_FEATURES:
        if f in PRICE_FEATURES: colors.append('steelblue')
        elif f in ISP_FEATURES: colors.append('darkorange')
        elif f in BM_FEATURES: colors.append('purple')
        else: colors.append('gray')

    sns.heatmap(corr, ax=ax, cmap='RdBu_r', center=0, vmin=-0.3, vmax=0.3,
                annot=True, fmt='.2f', annot_kws={'size': 7},
                yticklabels=[f.replace('_', ' , ', 1) if len(f) > 15 else f for f in ALL_FEATURES])

    # Color the y-axis labels
    for label, color in zip(ax.get_yticklabels(), colors):
        label.set_color(color)

    ax.set_title(f'{pname} (n={len(sub):,})', fontsize=11)
    ax.set_xticklabels(['IDA t+1h', 'IDA t+2h', 'IDAt+3h','Spread t+1h', 'Spread t+2h', 'Spreadt+3h'],rotation=0, fontsize=8)
    plt.suptitle('Feature Correlation with IDA Targets and IDA-DAM Spread''Blue=Price  Orange=ISP  Purple=BM  Gray=Other', fontsize=13, y=1.02)
plt.tight_layout(); plt.show()

## 2. ISP Revision Correlations — Focused Analysis

Correlation of **only ISP revision features** with the IDA-DAM spread, per horizon and per period. This is the key evidence that ISP revisions carry new information.

In [ ]:
# ISP revision correlations with spread — detailed table
REVISION_FEATURES = [
    'latest_RES_revision', 'latest_Load_revision', 'latest_net_load_revision',
    'RES_revision_ISP2_to_ISP3', 'Load_revision_ISP2_to_ISP3',
]

print("=" * 90)
print("  ISP Revision Features: Correlation with IDA-DAM Spread")
print("  (Pearson r, with p-values. Bold = statistically significant at p<0.01)")
print("=" * 90)

for pname, (d_start, d_end) in PERIODS.items():
    mask = (df['DELIVERY_MTU'] >= d_start) & (df['DELIVERY_MTU'] <= d_end + ' 23:45:00')
    sub = df[mask].dropna(subset=['spread_1h', 'spread_2h', 'spread_3h'])

    print(f" {pname} (n={len(sub):,})")
    print(f"  {'Feature':<32} {'r(spread_1h)':>14} {'r(spread_2h)':>14} {'r(spread_3h)':>14}")
    print(f"  {'-'*76}")

    for feat in REVISION_FEATURES:
        vals = []
        for sp in ['spread_1h', 'spread_2h', 'spread_3h']:
            r, p = stats.pearsonr(sub[feat].values, sub[sp].values)
            sig = '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else ''
            vals.append(f"{r:+.4f}{sig}")
        print(f"  {feat:<32} {vals[0]:>14} {vals[1]:>14} {vals[2]:>14}")

print(f" *** p<0.001  ** p<0.01  * p<0.05")

In [ ]:
# Visualization: ISP revision correlation bars per period
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

for ax, (pname, (d_start, d_end)) in zip(axes, PERIODS.items()):
    mask = (df['DELIVERY_MTU'] >= d_start) & (df['DELIVERY_MTU'] <= d_end + ' 23:45:00')
    sub = df[mask].dropna(subset=['spread_1h', 'spread_2h', 'spread_3h'])

    x = np.arange(len(REVISION_FEATURES))
    width = 0.25

    for h_idx, (sp, color, label) in enumerate([
        ('spread_1h', 'steelblue', 't+1h'),
        ('spread_2h', 'darkorange', 't+2h'),
        ('spread_3h', 'green', 't+3h')]):

        corrs = [stats.pearsonr(sub[f].values, sub[sp].values)[0] for f in REVISION_FEATURES]
        ax.bar(x + h_idx * width, corrs, width, color=color, alpha=0.7, label=label)

    ax.set_xticks(x + width)
    ax.set_xticklabels([f.replace('latest_', '').replace('_revision', 'rev').replace('_ISP2_to_ISP3', 'ISP2→3')
    for f in REVISION_FEATURES], fontsize=7)
    ax.set_ylabel('Pearson r')
    ax.set_title(pname)
    ax.axhline(0, color='black', linewidth=0.5)
    ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

plt.suptitle('ISP Revision Correlations with IDA-DAM Spread by Period', fontsize=13, y=1.02)
plt.tight_layout(); plt.show()

## 3. Mutual Information: Non-Linear Relationships

Pearson correlation only captures linear relationships. Mutual Information captures any dependency, including non-linear ones.

In [ ]:
from sklearn.feature_selection import mutual_info_regression

KEY_FEATURES = PRICE_FEATURES + REVISION_FEATURES + BM_FEATURES[:3] + ['Solar_DA_forecast', 'hour']
TARGET_HORIZONS = [('IDA_target_1h', 't+1h'), ('IDA_target_2h', 't+2h'), ('IDA_target_3h', 't+3h')]

fig, axes = plt.subplots(1, 3, figsize=(18, 7))

for ax, (pname, (d_start, d_end)) in zip(axes, PERIODS.items()):
    mask = (df['DELIVERY_MTU'] >= d_start) & (df['DELIVERY_MTU'] <= d_end + ' 23:45:00')
    sub = df[mask].dropna(subset=['IDA_target_1h']).copy()
    X = sub[KEY_FEATURES].fillna(0).values

    mi_results = {}
    for tgt_col, tgt_name in TARGET_HORIZONS:
        y = sub[tgt_col].values
        mi = mutual_info_regression(X, y, n_neighbors=5, random_state=42)
        mi_results[tgt_name] = mi

    x = np.arange(len(KEY_FEATURES))
    width = 0.25
    for h_idx, (tgt_name, color) in enumerate([('t+1h', 'steelblue'), ('t+2h', 'darkorange'), ('t+3h', 'green')]):
        ax.barh(x + h_idx * width, mi_results[tgt_name], width, color=color, alpha=0.7, label=tgt_name)

    # Color labels
    colors = []
    for f in KEY_FEATURES:
        if f in PRICE_FEATURES: colors.append('steelblue')
        elif f in REVISION_FEATURES: colors.append('darkorange')
        elif f in BM_FEATURES: colors.append('purple')
        else: colors.append('gray')

    ax.set_yticks(x + width)
    ax.set_yticklabels(KEY_FEATURES, fontsize=7)
    for label, c in zip(ax.get_yticklabels(), colors):
        label.set_color(c)
    ax.set_xlabel('Mutual Information')
    ax.set_title(pname, fontsize=10)
    ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

plt.suptitle(
    'Mutual Information: Features → IDA Target\nBlue=Price  Orange=ISP  Purple=BM  Gray=Other',
    fontsize=12,
    y=1.02,
)

plt.tight_layout(); plt.show()

## 4. Statistical Tests: ISP Features Add Information Beyond Prices

Partial correlation: Does `latest_net_load_revision` predict the spread even AFTER controlling for `MCP_IDA2` and `DAM_MCP`?

In [ ]:
def partial_corr(df, x, y, covariates):
    """Partial correlation of x and y, controlling for covariates."""
    from sklearn.linear_model import LinearRegression
    X_cov = df[covariates].values
    # Residualize x
    reg_x = LinearRegression().fit(X_cov, df[x].values)
    resid_x = df[x].values - reg_x.predict(X_cov)
    # Residualize y
    reg_y = LinearRegression().fit(X_cov, df[y].values)
    resid_y = df[y].values - reg_y.predict(X_cov)
    r, p = stats.pearsonr(resid_x, resid_y)
    return r, p

print("=" * 85)
print("  Partial Correlation: ISP Revision → Spread, controlling for Price Features")
print("  (Controls: MCP_IDA2 + DAM_MCP — the information already in the market)")
print("=" * 85)

CONTROLS = ['MCP_IDA2', 'DAM_MCP']

for pname, (d_start, d_end) in PERIODS.items():
    mask = (df['DELIVERY_MTU'] >= d_start) & (df['DELIVERY_MTU'] <= d_end + ' 23:45:00')
    sub = df[mask].dropna(subset=['spread_1h', 'spread_2h', 'spread_3h']).copy()

    print(f"{pname}")
    print(f"  {'Feature':<32} {'r_partial(1h)':>14} {'r_partial(2h)':>14} {'r_partial(3h)':>14}")
    print(f"  {'-'*76}")

    for feat in REVISION_FEATURES:
        vals = []
        for sp in ['spread_1h', 'spread_2h', 'spread_3h']:
            r, p = partial_corr(sub, feat, sp, CONTROLS)
            sig = '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else ''
            vals.append(f"{r:+.4f}{sig}")
        print(f"  {feat:<32} {vals[0]:>14} {vals[1]:>14} {vals[2]:>14}")

print(f"If partial r ≠ 0 and significant, ISP carries NEW information")
print(f"  beyond what prices already contain.")

## 5. Granger Causality: Do ISP Revisions Predict Future Spread Changes?

In [ ]:
from statsmodels.tsa.stattools import grangercausalitytests

print("=" * 70)
print("  Granger Causality: ISP Revision → IDA-DAM Spread")
print("  H0: ISP revision does NOT Granger-cause the spread")
print("  p < 0.05 means ISP revision has predictive power")
print("=" * 70)

for pname, (d_start, d_end) in PERIODS.items():
    mask = (df['DELIVERY_MTU'] >= d_start) & (df['DELIVERY_MTU'] <= d_end + ' 23:45:00')
    sub = df[mask].dropna(subset=['spread_1h']).copy()

    print(f"{pname}")
    for feat in ['latest_net_load_revision', 'latest_RES_revision', 'BM_IMBALANCE_PRICE']:
        series = sub[[feat, 'spread_1h']].dropna()
        try:
            result = grangercausalitytests(series[['spread_1h', feat]].values, maxlag=4, verbose=False)
            min_p = min(result[lag][0]['ssr_ftest'][1] for lag in range(1, 5))
            sig = '***' if min_p < 0.001 else '**' if min_p < 0.01 else '*' if min_p < 0.05 else 'ns'
            print(f"    {feat:<35} → spread: min_p={min_p:.6f} {sig}")
        except:
            print(f"    {feat:<35} → spread: ERROR (insufficient data)")

## 6. Summary: ISP Feature Evidence

This analysis provides four independent pieces of evidence:

1. **Pearson Correlation**: ISP revision features (especially `net_load_revision`) show significant correlation with the IDA-DAM spread across all periods and horizons.

2. **Mutual Information**: Non-linear dependencies confirm that ISP features carry information beyond linear relationships.

3. **Partial Correlation**: Even after controlling for current prices (MCP_IDA2, DAM_MCP), ISP revisions remain significantly correlated with the spread. This proves they carry **new** information not already expressed in market prices.

4. **Granger Causality**: ISP revisions Granger-cause future spread changes, confirming temporal predictive power.

Combined with the ablation study (ISP removal → +3-10% MAE increase), this constitutes strong evidence that ISP forecast revisions are a valuable feature set for IDA price forecasting.